# EnergAIzer — CPU Colab Demo

Fast, analytical **GPU power / energy / latency** prediction for AI workloads — no GPU required (runs on a free Colab **CPU** runtime).

Source: https://github.com/shubhamOjha1000/single_kernel_GPU_model (mirror of the ISPASS'26 EnergAIzer artifact).

Four use cases:
0. **Predict a single kernel** (one GEMM)
1. **Predict energy of an AI model** (BERT / ResNet / ViT …)
2. **Voltage–frequency scaling** (DVFS sweep on A100)
3. **Design-space exploration** (extrapolate A100 → H100)

> Run the 3 setup cells first (clone + install + download the LUT database), then any section. The Google Drive download is the one step that can occasionally fail due to quotas — see the note in that cell.

## Setup — Step 1: clone the repo + install CPU dependencies

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/shubhamOjha1000/single_kernel_GPU_model.git"
ROOT = "/content/single_kernel_GPU_model"
PKG  = os.path.join(ROOT, "energaizer-ispass26-artifact-main")

# 1) Clone the repo (the Python package `gee` lives inside PKG)
if not os.path.isdir(PKG):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], check=True)
print("Repo at:", PKG)

# 2) Install CPU-only deps. Inference from queries / workload JSONs does NOT need
#    torch or transformers (those are only for *generating* new workload JSONs).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "pandas", "scipy", "cvxpy", "pyyaml",
                "opt-einsum", "tqdm", "scikit-learn", "gdown"], check=True)
print("Dependencies installed.")

## Setup — Step 2: download the pre-collected LUT database

EnergAIzer needs an offline database of measured kernel latency/power (the lookup tables the analytical model is fitted against). It is **not** in the git repo and is downloaded from Google Drive (~tens of MB).

If `gdown` hits a Drive quota error, re-run this cell later, or download the file manually from the printed link and upload it to `PKG/database/data/`.

In [ ]:
import os, glob, subprocess

DB_BASE = os.path.join(PKG, "database", "data")
os.makedirs(DB_BASE, exist_ok=True)
DB_FILE_ID = "1krvqRFDnaqrJUT06V2psIua0wQr6ETAE"

def find_luts():
    return glob.glob(os.path.join(DB_BASE, "**", "*.csv"), recursive=True)

if not find_luts():
    archive = os.path.join(DB_BASE, "precollected_database.tar.gz")
    try:
        import gdown
        gdown.download(id=DB_FILE_ID, output=archive, quiet=False)
    except Exception as e:
        print("gdown failed:", e)
    if os.path.exists(archive):
        subprocess.run(["tar", "-xzf", archive, "-C", DB_BASE], check=True)
        os.remove(archive)
    else:
        # Fallback: the repo's curl-based downloader
        subprocess.run(["bash", os.path.join(PKG, "misc", "gdrive_download.sh"),
                        "https://drive.google.com/file/d/%s/view" % DB_FILE_ID],
                       cwd=DB_BASE, check=True)

luts = find_luts()
print("Found %d LUT csv files." % len(luts))
print("Manual link (if needed): https://drive.google.com/file/d/%s/view" % DB_FILE_ID)
assert luts, "LUT database not found - download failed. See the note above."

## Setup — Step 3: normalize LUT layout

The LUT config references CSVs by bare filename, resolved against `lut_folder_abs_path`. Ensure every CSV sits directly in `database/data/` (symlink up if the archive extracted into a subfolder).

In [ ]:
import os, glob

for csv in glob.glob(os.path.join(DB_BASE, "**", "*.csv"), recursive=True):
    dest = os.path.join(DB_BASE, os.path.basename(csv))
    if os.path.abspath(csv) != os.path.abspath(dest) and not os.path.exists(dest):
        os.symlink(os.path.abspath(csv), dest)

flat = sorted(f for f in os.listdir(DB_BASE) if f.endswith(".csv"))
print("%d CSVs directly in database/data:" % len(flat))
for f in flat:
    print("  ", f)

## 0. Predict a single kernel

Build the estimator once (loads all lookup tables), then query a single **bfloat16 Tensor-Core GEMM** on an A100-40GB-PCIE (`yz8`) at 900 MHz. Returns `(latency, power, energy)`.

In [ ]:
import sys, os
if PKG not in sys.path:
    sys.path.insert(0, PKG)

from gee import get_gee

A100_GPU   = os.path.join(PKG, "config", "gpu", "yz8.yaml")          # yz8 == A100-40GB-PCIE
LUT_CONFIG = os.path.join(PKG, "experiments_endtoend", "exp_config", "a100_lut_config.yaml")
VOLT_JSON  = os.path.join(PKG, "config", "dvfs", "yz8", "supply_voltage.json")
IDLE_JSON  = os.path.join(PKG, "config", "dvfs", "yz8", "idle_power.json")

estimator = get_gee(
    gpu_yaml_path=A100_GPU,
    lut_yaml_path=LUT_CONFIG,
    dvfs_aware=True,
    dvfs_inference_mode="all",
    dvfs_supply_voltage_json=VOLT_JSON,
    dvfs_idle_power_json=IDLE_JSON,
    lut_folder_abs_path=DB_BASE,
)
print("Estimator built.")

In [ ]:
# A single bf16 Tensor-Core GEMM:  (batch x M x K) @ (batch x K x N)
query = {
    "batch": 1,
    "dimM": 4096,
    "dimN": 4096,
    "dimK": 4096,
    "precM": "bf16",        # operand precision
    "precA": "bf16",        # accumulate / output precision
    "useTensorCore": True,
}
query_type = ("gemm",
              "tc" if query["useTensorCore"] else "cuda",
              "{}_{}".format(query["precM"], query["precA"]))

latency, power, energy = estimator.lookup(query, query_type,
                                          target_freq=900, lookup_target="all")

print("GEMM  batch=1, M=N=K=4096  (bf16, Tensor Core)  @ A100, 900 MHz")
print("  latency :", latency)
print("  power   :", power, "W")
print("  energy  :", energy)

## 1. Predict energy consumption of an AI model

Run the end-to-end estimator over a few provided workload JSONs (BERT, ResNet101, ViT). Each JSON is a decomposed list of kernels for a model+shape. Prior-work baselines are disabled (`--no_neusight/--no_limicro/--no_roofline`) since they need extra downloads. Output is a CSV with per-workload latency and energy.

> A small subset is used to keep it fast — point `--workload_folder` at `test/data/workloads/all` for the full set.

In [ ]:
import os, shutil, subprocess
import pandas as pd

ENDTOEND = os.path.join(PKG, "experiments_endtoend")

SRC = os.path.join(PKG, "test", "data", "workloads", "all")
SUB = os.path.join(PKG, "test", "data", "workloads", "demo_subset")
os.makedirs(SUB, exist_ok=True)
for p in ["bertmodel_large_pbf16_b8_s128_modeprefill.json",
          "resnet101__pbf16_b8_freq900.json",
          "vitmodel_large_pbf16_b16_freq900.json"]:
    src = os.path.join(SRC, p)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SUB, p))
print("Subset workloads:", os.listdir(SUB))

cmd = [sys.executable, "run_estimators.py",
       "--workload_folder", SUB,
       "--result_save_to", "results/a100_endtoend",
       "--result_filename", "demo.csv",
       "--gpu_config_yaml", "../config/gpu/yz8.yaml",
       "--lut_config_yaml", "exp_config/a100_lut_config.yaml",
       "--lut_folder_abs_path", DB_BASE,
       "--no_neusight", "--no_limicro", "--no_roofline",
       "--target_freq", "900",
       "--flash_attention_enable",
       "--flash_attention_estimate_method", "flashattention_v2"]
subprocess.run(cmd, cwd=ENDTOEND, check=True)

pd.set_option("display.max_columns", None)
df = pd.read_csv(os.path.join(ENDTOEND, "results/a100_endtoend/demo.csv"))
df

## 2. Voltage–frequency scaling (DVFS)

Same estimator, but `--dvfs_aware` sweeps the core frequency 510 → 1410 MHz (using the A100 supply-voltage and idle-power profiles). The output shows how predicted latency/power/energy change with frequency for each workload.

In [ ]:
import os, shutil, subprocess
import pandas as pd

DVFS_SRC = os.path.join(PKG, "test", "data", "workloads", "dvfs")
DVFS_SUB = os.path.join(PKG, "test", "data", "workloads", "dvfs_demo")
os.makedirs(DVFS_SUB, exist_ok=True)
for p in ["gpt2model_gpt2_pbf16_b8_s128_modeprefill.json",
          "optmodel_opt-1p3b_pbf16_b2_s128_modeprefill.json"]:
    src = os.path.join(DVFS_SRC, p)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DVFS_SUB, p))
print("DVFS workloads:", os.listdir(DVFS_SUB))

cmd = [sys.executable, "run_estimators.py",
       "--workload_folder", DVFS_SUB,
       "--result_save_to", "results/a100_dvfs",
       "--result_filename", "dvfs_demo.csv",
       "--gpu_config_yaml", "../config/gpu/yz8.yaml",
       "--lut_config_yaml", "exp_config/a100_dvfs_lut_config.yaml",
       "--lut_folder_abs_path", DB_BASE,
       "--no_neusight", "--no_limicro", "--no_roofline",
       "--dvfs_aware",
       "--dvfs_supply_voltage_json", "../config/dvfs/yz8/supply_voltage.json",
       "--dvfs_idle_power_json", "../config/dvfs/yz8/idle_power.json",
       "--freq_min", "510", "--freq_max", "1410", "--freq_step", "180",
       "--flash_attention_enable",
       "--flash_attention_estimate_method", "flashattention_v2"]
subprocess.run(cmd, cwd=ENDTOEND, check=True)

df = pd.read_csv(os.path.join(ENDTOEND, "results/a100_dvfs/dvfs_demo.csv"))
df

## 3. Design-space exploration (A100 → H100 extrapolation)

EnergAIzer can forecast a *different* GPU architecture by re-using the A100-fitted hardware constants and scaling by the target GPU's SM count / bandwidth / Tensor-Core throughput. Here we extrapolate to an **H100-SXM @ 1830 MHz, 1.0 V** with no H100 measurements.

In [ ]:
import os, subprocess
import pandas as pd

cmd = [sys.executable, "run_estimators.py",
       "--workload_folder", SUB,                       # reuse the subset from section 1
       "--result_save_to", "results/h100_extrap",
       "--result_filename", "h100_demo.csv",
       "--multiple_config",
       "--multiple_gpu_configs_yaml", "../config/multiple/gpu.yaml",
       "--multiple_gpu_supply_voltage_yaml", "../config/multiple/supply_voltage.yaml",
       "--multiple_gpu_idle_power_yaml", "../config/multiple/idle_power.yaml",
       "--lut_config_yaml", "exp_config/extrapolation_lut_config.yaml",
       "--lut_folder_abs_path", DB_BASE,
       "--no_neusight", "--no_limicro", "--no_roofline",
       "--extrapolation",
       "--target_gpu_config_yaml", "../config/gpu/h100sxm.yaml",
       "--target_supply_voltage_value", "1000",
       "--target_freq", "1830",
       "--target_supply_voltage_json", "../config/dvfs/h100/supply_voltage.json",
       "--target_idle_power_json", "../config/dvfs/h100/idle_power.json",
       "--flash_attention_enable",
       "--flash_attention_estimate_method", "flashattention_v2"]
subprocess.run(cmd, cwd=ENDTOEND, check=True)

df = pd.read_csv(os.path.join(ENDTOEND, "results/h100_extrap/h100_demo.csv"))
df

## 4. PCIe vs SXM — the latency / power / energy crossover

Compare the **same bf16 Tensor-Core GEMM** on the two A100 variants in the database — identical 312-TFLOPs / 108-SM silicon, differing only in memory bandwidth and power/voltage profile:

- `yz8` = A100-40GB-**PCIe**   (DRAM 1555 GB/s, ~250 W cap)
- `netsres117` = A100-80GB-**SXM**   (DRAM 2039 GB/s, ~400 W cap)

We fix M=N=4096 and sweep the contraction dim K to move along the **arithmetic-intensity** axis (low K = memory-bound, high K = compute-bound; the A100 bf16 roofline ridge is ~200 FLOP/byte). Both are predicted at the **same 900 MHz**, so the only differences are bandwidth + the per-node voltage/idle profile.

Watch the `E_SXM/E_PCIe` column cross 1.0: **below 1.0 SXM wins** (memory-bound — its bandwidth cuts time enough to offset higher power); **above 1.0 PCIe wins** (compute-bound — same speed, but SXM burns more watts).

> Single-GPU only — this is the on-package bandwidth/power tradeoff, **not** the NVLink-vs-PCIe interconnect question.

In [ ]:
import os, yaml
from gee import get_gee

# Build a LUT config for the A100-SXM (netsres117) by reusing the A100-PCIe config
# and swapping in the netsres GEMM tables (GEMM is the only kernel we query here;
# the borrowed softmax/conv/etc. tables are never exercised).
ENDTOEND = os.path.join(PKG, "experiments_endtoend")
A100_CFG = os.path.join(ENDTOEND, "exp_config", "a100_lut_config.yaml")
NET_CFG  = os.path.join(ENDTOEND, "exp_config", "netsres117_lut_config.yaml")

cfg = yaml.safe_load(open(A100_CFG))
cfg["lut_config"]["gemm"] = {
    "tc":   {"bf16_bf16": [dict(path="netsres117_gemm_bf16_bf16_lut_prepared.csv",
                                prepared=True, main=True, use_for_model=True,
                                use_for_kernel=True, require_annotation=False)]},
    "cuda": {"fp32_fp32": [dict(path="netsres117_sgemm_fp32_lut_v2_prepared.csv",
                                prepared=True, main=True, use_for_model=True,
                                use_for_kernel=True, require_annotation=False)]},
}
with open(NET_CFG, "w") as f:
    yaml.safe_dump(cfg, f)
print("Wrote", NET_CFG)

def build_estimator(gpu_name, lut_yaml):
    return get_gee(
        gpu_yaml_path=os.path.join(PKG, "config", "gpu", gpu_name + ".yaml"),
        lut_yaml_path=lut_yaml,
        dvfs_aware=True, dvfs_inference_mode="all",
        dvfs_supply_voltage_json=os.path.join(PKG, "config", "dvfs", gpu_name, "supply_voltage.json"),
        dvfs_idle_power_json=os.path.join(PKG, "config", "dvfs", gpu_name, "idle_power.json"),
        lut_folder_abs_path=DB_BASE,
    )

pcie_est = build_estimator("yz8",        A100_CFG)   # A100-40GB-PCIe, 1555 GB/s
sxm_est  = build_estimator("netsres117", NET_CFG)    # A100-80GB-SXM,  2039 GB/s
print("Both estimators built.")

In [ ]:
import pandas as pd

FREQ = 900
M = N = 4096
Ks = [32, 128, 512, 2048, 4096]
qtype = ("gemm", "tc", "bf16_bf16")

rows = []
for K in Ks:
    q = {"batch": 1, "dimM": M, "dimN": N, "dimK": K,
         "precM": "bf16", "precA": "bf16", "useTensorCore": True}
    flop = 2 * M * N * K
    byts = (M * N + M * K + N * K) * 2          # bf16 = 2 bytes/element
    ai   = flop / byts                          # arithmetic intensity (FLOP/byte)
    tp, pp, ep = pcie_est.lookup(q, qtype, target_freq=FREQ, lookup_target="all")
    ts, ps, es = sxm_est.lookup(q,  qtype, target_freq=FREQ, lookup_target="all")
    rows.append({
        "K": K, "intensity": round(ai, 1),
        "lat_PCIe_ms": round(float(tp), 4), "lat_SXM_ms": round(float(ts), 4),
        "pow_PCIe_W": round(float(pp), 1),  "pow_SXM_W": round(float(ps), 1),
        "E_PCIe_J": round(float(ep), 4),    "E_SXM_J": round(float(es), 4),
        "E_SXM/E_PCIe": round(float(es) / float(ep), 3),
    })

df = pd.DataFrame(rows)
print("A100  M=N=%d  bf16 Tensor-Core GEMM @ %d MHz   (roofline ridge ~200 FLOP/byte)" % (M, FREQ))
print("E_SXM/E_PCIe < 1 -> SXM more energy-efficient;  > 1 -> PCIe more energy-efficient\n")
df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(df["intensity"], df["lat_PCIe_ms"], "o-", label="PCIe (1555 GB/s)")
ax[0].plot(df["intensity"], df["lat_SXM_ms"], "s-", label="SXM (2039 GB/s)")
ax[0].set_xscale("log"); ax[0].set_xlabel("Arithmetic intensity (FLOP/byte)")
ax[0].set_ylabel("Latency (ms)"); ax[0].set_title("Latency: SXM faster when memory-bound")
ax[0].legend(); ax[0].grid(True, which="both", alpha=0.3)

ax[1].axhline(1.0, ls="--", c="gray")
ax[1].plot(df["intensity"], df["E_SXM/E_PCIe"], "o-", c="tab:red")
ax[1].set_xscale("log"); ax[1].set_xlabel("Arithmetic intensity (FLOP/byte)")
ax[1].set_ylabel("Energy ratio  SXM / PCIe")
ax[1].set_title("Energy crossover (below 1.0 = SXM wins)")
ax[1].grid(True, which="both", alpha=0.3)

plt.tight_layout(); plt.show()

## 5. Compute-bound vs memory-bound: where SXM's bandwidth actually wins

Section 4 showed square GEMMs give identical latency on both A100s (compute / L2-bound — the only differing spec, DRAM bandwidth, barely matters). To make latency *diverge*, we need **DRAM-bound** kernels with little L2 reuse. So we compare three kernels of very different arithmetic intensity, using the **real netsres (SXM) tables** for each:

| Kernel | Shape | Type | Expectation |
|---|---|---|---|
| GEMM | 4096x4096x4096 | **compute-bound** (~1365 FLOP/byte) | latency identical; SXM more power |
| Softmax | 8192 x 8192 | **memory-bound** (streaming, > L2) | SXM faster (more bandwidth) |
| LayerNorm | 8192 x 8192 | **memory-bound** (streaming, > L2) | SXM faster (more bandwidth) |

`lat_speedup = lat_PCIe / lat_SXM` (>1 means SXM is faster). Both predicted at 900 MHz so only the hardware differs.

> Note: the netsres softmax/layernorm tables were measured unlocked (~1410 MHz) and are scaled to 900 MHz by the DVFS model; treat the memory-bound numbers as directional.

In [ ]:
import os, yaml
from gee import get_gee

ENDTOEND = os.path.join(PKG, "experiments_endtoend")
A100_CFG = os.path.join(ENDTOEND, "exp_config", "a100_lut_config.yaml")
NET_CFG2 = os.path.join(ENDTOEND, "exp_config", "netsres117_lut_config_v2.yaml")

# SXM config: real netsres tables for GEMM (compute) + softmax & layernorm (memory).
cfg = yaml.safe_load(open(A100_CFG))
lc = cfg["lut_config"]
lc["gemm"] = {
    "tc":   {"bf16_bf16": [dict(path="netsres117_gemm_bf16_bf16_lut_prepared.csv",
                                prepared=True, main=True, use_for_model=True,
                                use_for_kernel=True, require_annotation=False)]},
    "cuda": {"fp32_fp32": [dict(path="netsres117_sgemm_fp32_lut_v2_prepared.csv",
                                prepared=True, main=True, use_for_model=True,
                                use_for_kernel=True, require_annotation=False)]},
}
# Swap only the bf16 path of softmax/layernorm to the real netsres (SXM) tables;
# keep every flag (incl. the upstream 'prepaared' typo) intact to match the loader.
lc["softmax"]["bf16"][0]["path"]   = "netsres_softmax_bf16_nolock.csv"
lc["layernorm"]["bf16"][0]["path"] = "netsres_layernorm_bf16_nolock.csv"
with open(NET_CFG2, "w") as f:
    yaml.safe_dump(cfg, f)
print("Wrote", NET_CFG2)

def build_estimator(gpu_name, lut_yaml):
    return get_gee(
        gpu_yaml_path=os.path.join(PKG, "config", "gpu", gpu_name + ".yaml"),
        lut_yaml_path=lut_yaml,
        dvfs_aware=True, dvfs_inference_mode="all",
        dvfs_supply_voltage_json=os.path.join(PKG, "config", "dvfs", gpu_name, "supply_voltage.json"),
        dvfs_idle_power_json=os.path.join(PKG, "config", "dvfs", gpu_name, "idle_power.json"),
        lut_folder_abs_path=DB_BASE,
    )

pcie_est2 = build_estimator("yz8",        A100_CFG)   # A100-40GB-PCIe, 1555 GB/s
sxm_est2  = build_estimator("netsres117", NET_CFG2)   # A100-80GB-SXM,  2039 GB/s
print("Both estimators built.")

In [ ]:
import pandas as pd

FREQ = 900

def run(est, q, qt):
    t, p, e = est.lookup(q, qt, target_freq=FREQ, lookup_target="all")
    return float(t), float(p), float(e)

kernels = [
    ("GEMM 4096^3 (compute)",  {"batch":1,"dimM":4096,"dimN":4096,"dimK":4096,
                                "precM":"bf16","precA":"bf16","useTensorCore":True},
                               ("gemm","tc","bf16_bf16")),
    ("Softmax 8192x8192 (mem)", {"batch":8192,"dim":8192,"prec":"bf16"}, ("softmax","bf16")),
    ("LayerNorm 8192x8192 (mem)", {"batch":8192,"dim":8192,"prec":"bf16"}, ("layernorm","bf16")),
]

rows = []
for name, q, qt in kernels:
    tp, pp, ep = run(pcie_est2, q, qt)
    ts, ps, es = run(sxm_est2,  q, qt)
    rows.append({
        "kernel": name,
        "lat_PCIe_ms": round(tp,4), "lat_SXM_ms": round(ts,4),
        "lat_speedup": round(tp/ts,3),
        "pow_PCIe_W": round(pp,1), "pow_SXM_W": round(ps,1),
        "E_PCIe_J": round(ep,4),   "E_SXM_J": round(es,4),
        "E_SXM/E_PCIe": round(es/ep,3),
    })

df5 = pd.DataFrame(rows)
print("A100 PCIe vs SXM @ %d MHz   (lat_speedup>1 => SXM faster; E ratio<1 => SXM lower energy)\n" % FREQ)
df5

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(df5)); w = 0.38
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

ax[0].bar(x-w/2, df5["lat_PCIe_ms"], w, label="PCIe")
ax[0].bar(x+w/2, df5["lat_SXM_ms"], w, label="SXM")
ax[0].set_yscale("log"); ax[0].set_ylabel("Latency (ms, log)")
ax[0].set_xticks(x); ax[0].set_xticklabels(df5["kernel"], rotation=20, ha="right")
ax[0].set_title("Latency: identical for compute, SXM wins for memory"); ax[0].legend()

ax[1].bar(x-w/2, df5["E_PCIe_J"], w, label="PCIe")
ax[1].bar(x+w/2, df5["E_SXM_J"], w, label="SXM")
ax[1].set_yscale("log"); ax[1].set_ylabel("Energy (J, log)")
ax[1].set_xticks(x); ax[1].set_xticklabels(df5["kernel"], rotation=20, ha="right")
ax[1].set_title("Energy per kernel"); ax[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd

FREQ = 900

def run(est, q, qt):
    t, p, e = est.lookup(q, qt, target_freq=FREQ, lookup_target="all")
    return float(t), float(p), float(e)

kernels = [
    ("GEMM 4096^3 (compute)",  {"batch":1,"dimM":4096,"dimN":4096,"dimK":4096,
                                "precM":"bf16","precA":"bf16","useTensorCore":True},
                               ("gemm","tc","bf16_bf16")),
    ("Softmax 8192x8192 (mem)", {"batch":1,"dim":4096,"prec":"bf16"}, ("softmax","bf16")),
    ("LayerNorm 8192x8192 (mem)", {"batch":1,"dim":4096,"prec":"bf16"}, ("layernorm","bf16")),
]

rows = []
for name, q, qt in kernels:
    tp, pp, ep = run(pcie_est2, q, qt)
    ts, ps, es = run(sxm_est2,  q, qt)
    rows.append({
        "kernel": name,
        "lat_PCIe_ms": round(tp,4), "lat_SXM_ms": round(ts,4),
        "lat_speedup": round(tp/ts,3),
        "pow_PCIe_W": round(pp,1), "pow_SXM_W": round(ps,1),
        "E_PCIe_J": round(ep,4),   "E_SXM_J": round(es,4),
        "E_SXM/E_PCIe": round(es/ep,3),
    })

df5 = pd.DataFrame(rows)
print("A100 PCIe vs SXM @ %d MHz   (lat_speedup>1 => SXM faster; E ratio<1 => SXM lower energy)\n" % FREQ)
df5

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(df5)); w = 0.38
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

ax[0].bar(x-w/2, df5["lat_PCIe_ms"], w, label="PCIe")
ax[0].bar(x+w/2, df5["lat_SXM_ms"], w, label="SXM")
ax[0].set_yscale("log"); ax[0].set_ylabel("Latency (ms, log)")
ax[0].set_xticks(x); ax[0].set_xticklabels(df5["kernel"], rotation=20, ha="right")
ax[0].set_title("Latency: identical for compute, SXM wins for memory"); ax[0].legend()

ax[1].bar(x-w/2, df5["E_PCIe_J"], w, label="PCIe")
ax[1].bar(x+w/2, df5["E_SXM_J"], w, label="SXM")
ax[1].set_yscale("log"); ax[1].set_ylabel("Energy (J, log)")
ax[1].set_xticks(x); ax[1].set_xticklabels(df5["kernel"], rotation=20, ha="right")
ax[1].set_title("Energy per kernel"); ax[1].legend()

plt.tight_layout(); plt.show()

## 6. Isolating bandwidth: same LUT, sweep only `dram_bw`

The controlled experiment. We hold **everything** fixed — the empirical anchor (yz8 freq900 tables), the voltage & idle-power profile, SMs, clock, Tensor Cores, L2 — and change **only** `dram_bw`:

- 1555 GB/s  (A100-PCIe)
- 2039 GB/s  (A100-SXM)
- 3352 GB/s  (H100-class)

Because the LUT is identical for all three, the empirical correction `(lambda, epsilon)` is identical too, so **any latency change is purely the model's analytical response to bandwidth** — no measurement-condition confound.

**What to look for:**
- **GEMM (compute-bound)** -> latency should stay ~flat (the control; its time is set by Tensor Cores / L2, not DRAM).
- **Softmax / LayerNorm (memory-bound)** -> if truly DRAM-bound, latency should fall toward the ideal `1555/bw` (≈0.76x at 2039, ≈0.46x at 3352). If it barely moves, the model's memory-bound time is dominated by the L2 path / fixed overhead instead.

> Isolates the bandwidth knob alone (`dram_freq` held constant); this is the model's *sensitivity*, not a real measurement.

In [ ]:
import os, yaml, copy
from gee import get_gee

BASE_GPU = os.path.join(PKG, "config", "gpu", "yz8.yaml")
A100_CFG = os.path.join(PKG, "experiments_endtoend", "exp_config", "a100_lut_config.yaml")
VOLT = os.path.join(PKG, "config", "dvfs", "yz8", "supply_voltage.json")
IDLE = os.path.join(PKG, "config", "dvfs", "yz8", "idle_power.json")

base = yaml.safe_load(open(BASE_GPU))
BWS = [1555, 2039, 3352]   # PCIe, SXM, H100-class

ests = {}
for bw in BWS:
    cfg = copy.deepcopy(base)
    cfg["gpu_configs"]["dram_bw"] = bw          # change ONLY this
    path = os.path.join(PKG, "config", "gpu", "yz8_bw%d.yaml" % bw)
    with open(path, "w") as f:
        yaml.safe_dump(cfg, f)
    ests[bw] = get_gee(
        gpu_yaml_path=path, lut_yaml_path=A100_CFG,
        dvfs_aware=True, dvfs_inference_mode="all",
        dvfs_supply_voltage_json=VOLT, dvfs_idle_power_json=IDLE,
        lut_folder_abs_path=DB_BASE,
    )
print("Built estimators for dram_bw =", list(ests))

In [ ]:
import pandas as pd

FREQ = 900
kernels = [
    ("GEMM 4096^3 (compute)",   {"batch":1,"dimM":4096,"dimN":4096,"dimK":4096,
                                 "precM":"bf16","precA":"bf16","useTensorCore":True}, ("gemm","tc","bf16_bf16")),
    ("Softmax 8192x8192 (mem)",   {"batch":8192,"dim":8192,"prec":"bf16"}, ("softmax","bf16")),
    ("LayerNorm 8192x8192 (mem)", {"batch":8192,"dim":8192,"prec":"bf16"}, ("layernorm","bf16")),
]

rows = []
for name, q, qt in kernels:
    base_t = None
    for bw in BWS:
        t, p, e = ests[bw].lookup(q, qt, target_freq=FREQ, lookup_target="all")
        t, p, e = float(t), float(p), float(e)
        if base_t is None:
            base_t = t
        rows.append({"kernel":name, "dram_bw":bw, "lat_ms":round(t,4),
                     "lat_vs_1555":round(t/base_t,3), "ideal_1555/bw":round(1555.0/bw,3),
                     "pow_W":round(p,1), "E_J":round(e,5)})

df6 = pd.DataFrame(rows)
print("Same yz8 LUT + voltage; only dram_bw varies. @ %d MHz\n" % FREQ)
df6

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4.5))
for name in df6["kernel"].unique():
    sub = df6[df6["kernel"] == name]
    plt.plot(sub["dram_bw"], sub["lat_vs_1555"], "o-", label=name)
# ideal memory-bound scaling 1555/bw
ideal = df6[df6["kernel"]==df6["kernel"].iloc[0]]
plt.plot(ideal["dram_bw"], 1555.0/ideal["dram_bw"], "k--", alpha=0.6, label="ideal mem-bound (1555/bw)")
plt.xlabel("DRAM bandwidth (GB/s)"); plt.ylabel("Latency relative to 1555 GB/s")
plt.title("Isolating bandwidth: compute-bound flat, memory-bound should drop")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 7. The right way: extrapolation mode across 3 GPUs

Section 6 proved you cannot probe bandwidth by editing `dram_bw` (the empirical fit re-absorbs it). The framework's **extrapolation mode** is the correct tool: it builds **one source model** (fit on the yz8 + netsres117 measured data) and **scales** it onto each target GPU config (bandwidth, FLOPs, SMs) — no per-GPU refit, so no LUT-anchor confound.

We drive the proven `run_estimators.py --extrapolation` engine (same as Section 3) over three single-kernel workloads, retargeting three GPUs at the **same 900 MHz**:

| GPU | bf16 TFLOPs | DRAM GB/s | character |
|---|---|---|---|
| A100-PCIe | 312 | 1555 | baseline |
| A100-SXM | 312 | 2039 | same compute, +31% bandwidth |
| H100-SXM | 989 | 3352 | +3.2x compute, +2.2x bandwidth |

Each GPU's **900 MHz supply voltage is pinned as a constant** (`--target_supply_voltage_value`). The A100s use their measured 900 MHz voltage (694 / 744 mV); the **H100 voltage table is uncharacterized (all null) in this dataset**, so we assume ~750 mV — meaning H100 **latency** is solid but its **power/energy is approximate**.

**Expected:** GEMM (compute-bound) speeds up with **TFLOPs** (H100 fastest; the two A100s ~equal). Softmax/LayerNorm (memory-bound) speed up with **bandwidth** (PCIe < SXM < H100) — where A100-SXM should finally beat A100-PCIe, the effect Section 5 masked.

In [ ]:
import os, json

# One single-kernel workload file per kernel (run_estimators sums per-file, so 1 entry = that kernel).
WL = os.path.join(PKG, "test", "data", "workloads", "compare3gpu")
os.makedirs(WL, exist_ok=True)
specs = {
    "gemm_compute.json":  [[{"batch":1,"dimM":4096,"dimN":4096,"dimK":4096,
                             "precM":"bf16","precA":"bf16","useTensorCore":True}, ["gemm","tc","bf16_bf16"]]],
    "softmax_mem.json":   [[{"batch":8192,"dim":8192,"prec":"bf16"}, ["softmax","bf16"]]],
    "layernorm_mem.json": [[{"batch":8192,"dim":8192,"prec":"bf16"}, ["layernorm","bf16"]]],
}
for fn, content in specs.items():
    with open(os.path.join(WL, fn), "w") as f:
        json.dump(content, f)
print("workloads:", os.listdir(WL))

In [ ]:
import os, json, subprocess
import pandas as pd

ENDTOEND = os.path.join(PKG, "experiments_endtoend")
FREQ = 900

# display name -> (gpu config yaml stem, dvfs json folder)
targets = {
    "A100-PCIe": ("yz8",        "yz8"),
    "A100-SXM":  ("netsres117", "netsres117"),
    "H100-SXM":  ("h100sxm",    "h100"),
}

frames = []
for gname, (stem, dvfs) in targets.items():
    v900 = json.load(open(os.path.join(PKG, "config", "dvfs", dvfs, "supply_voltage.json"))).get(str(FREQ))
    if v900 is None:
        v900 = 750.0   # H100 supply-voltage table is uncharacterized (all null) in this DB;
                       # assume ~750 mV at 900 MHz (comparable to the A100-SXM floor). Power/energy for H100 is approximate.
    out = "results/compare3gpu"
    cmd = ["python", "run_estimators.py",
           "--workload_folder", WL,
           "--result_save_to", out, "--result_filename", stem + ".csv",
           "--multiple_config",
           "--multiple_gpu_configs_yaml", "../config/multiple/gpu.yaml",
           "--multiple_gpu_supply_voltage_yaml", "../config/multiple/supply_voltage.yaml",
           "--multiple_gpu_idle_power_yaml", "../config/multiple/idle_power.yaml",
           "--lut_config_yaml", "exp_config/extrapolation_lut_config.yaml",
           "--lut_folder_abs_path", DB_BASE,
           "--no_neusight", "--no_limicro", "--no_roofline",
           "--extrapolation",
           "--target_gpu_config_yaml", "../config/gpu/%s.yaml" % stem,
           "--target_supply_voltage_value", str(v900),
           "--target_freq", str(FREQ),
           "--target_supply_voltage_json", "../config/dvfs/%s/supply_voltage.json" % dvfs,
           "--target_idle_power_json", "../config/dvfs/%s/idle_power.json" % dvfs,
           "--flash_attention_enable", "--flash_attention_estimate_method", "flashattention_v2"]
    subprocess.run(cmd, cwd=ENDTOEND, check=True)
    d = pd.read_csv(os.path.join(ENDTOEND, out, stem + ".csv"))
    d["gpu"] = gname
    frames.append(d)

df7 = pd.concat(frames, ignore_index=True)
df7["power_W"] = (df7["energy_predicted"] / df7["time_predicted"] * 1000).round(1)
df7 = df7.rename(columns={"time_predicted":"lat_ms", "energy_predicted":"E_J"})
df7 = df7[["workload", "gpu", "lat_ms", "power_W", "E_J"]]
print("Extrapolation across 3 GPUs @ %d MHz (one source model scaled to each target)\n" % FREQ)
df7

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

order = list(df7["workload"].unique()); gpus = list(targets.keys())
x = np.arange(len(order)); w = 0.25
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for j, g in enumerate(gpus):
    sub = df7[df7["gpu"]==g].set_index("workload").loc[order]
    ax[0].bar(x+(j-1)*w, sub["lat_ms"], w, label=g)
    ax[1].bar(x+(j-1)*w, sub["E_J"],   w, label=g)
for a, ttl, yl in ((ax[0],"Latency","ms"), (ax[1],"Energy","J")):
    a.set_yscale("log"); a.set_xticks(x); a.set_xticklabels(order, rotation=15, ha="right")
    a.set_ylabel("%s (%s, log)" % (ttl, yl)); a.set_title(ttl + " by GPU"); a.legend()
plt.tight_layout(); plt.show()

## 8. Core-clock DVFS: compute-bound vs memory-bound

On **A100-PCIe (`yz8`)** we sweep the **core/SM clock** (510 / 705 / 900 MHz) with the **DRAM clock fixed**, and measure latency / power / energy for two buckets:

- **Compute-bound:** GEMM 4096^3 and 2048^3 (bf16 Tensor Core) — capped at 900 MHz to stay on locked measurements.
- **Memory-bound:** Softmax 8192x8192, LayerNorm 8192x8192, Elementwise (64M adds).

**Hypothesis:** raising the core clock helps compute-bound kernels (latency ~ 1/freq) but **not** memory-bound ones (latency pinned by DRAM bandwidth, which the core clock doesn't touch) — while **power rises for both**. So clocking up is worth it for compute, wasteful for memory.

_Note: on this GPU the supply voltage is constant (~694 mV) across 510-1020 MHz, so in this range the effect is **pure frequency** (no V^2 penalty yet) — dynamic power rises ~linearly with clock, which makes the compute-vs-memory contrast clean. Expect: compute latency falls (~flat energy), memory latency flat (energy rises = wasted clock)._

In [ ]:
import os, json

WL8 = os.path.join(PKG, "test", "data", "workloads", "dvfs_compute_vs_mem")
os.makedirs(WL8, exist_ok=True)
specs = {
    # compute-bound
    "gemm_4096.json": [[{"batch":1,"dimM":4096,"dimN":4096,"dimK":4096,
                         "precM":"bf16","precA":"bf16","useTensorCore":True}, ["gemm","tc","bf16_bf16"]]],
    "gemm_2048.json": [[{"batch":1,"dimM":2048,"dimN":2048,"dimK":2048,
                         "precM":"bf16","precA":"bf16","useTensorCore":True}, ["gemm","tc","bf16_bf16"]]],
    # memory-bound
    "softmax.json":     [[{"batch":8192,"dim":8192,"prec":"bf16"}, ["softmax","bf16"]]],
    "layernorm.json":   [[{"batch":8192,"dim":8192,"prec":"bf16"}, ["layernorm","bf16"]]],
    "elementwise.json": [[{"dim":67108864,"op":"pointwise_add","prec":"bf16"}, ["elementwise"]]],
}
for fn, content in specs.items():
    with open(os.path.join(WL8, fn), "w") as f:
        json.dump(content, f)
print("workloads:", os.listdir(WL8))

In [ ]:
import os, subprocess
import pandas as pd

ENDTOEND = os.path.join(PKG, "experiments_endtoend")
out = "results/dvfs_compute_vs_mem"
cmd = ["python", "run_estimators.py",
       "--workload_folder", WL8,
       "--result_save_to", out, "--result_filename", "dvfs.csv",
       "--gpu_config_yaml", "../config/gpu/yz8.yaml",
       "--lut_config_yaml", "exp_config/a100_dvfs_lut_config.yaml",
       "--lut_folder_abs_path", DB_BASE,
       "--no_neusight", "--no_limicro", "--no_roofline",
       "--dvfs_aware",
       "--dvfs_supply_voltage_json", "../config/dvfs/yz8/supply_voltage.json",
       "--dvfs_idle_power_json", "../config/dvfs/yz8/idle_power.json",
       "--freq_min", "510", "--freq_max", "900", "--freq_step", "195",
       "--flash_attention_enable", "--flash_attention_estimate_method", "flashattention_v2"]
subprocess.run(cmd, cwd=ENDTOEND, check=True)

df8 = pd.read_csv(os.path.join(ENDTOEND, out, "dvfs.csv"))
bucket = {"gemm_4096.json":"compute", "gemm_2048.json":"compute",
          "softmax.json":"memory", "layernorm.json":"memory", "elementwise.json":"memory"}
df8["bucket"] = df8["workload"].map(bucket)
df8["kernel"] = df8["workload"].str.replace(".json", "", regex=False)
df8["power_W"] = (df8["energy_predicted"] / df8["time_predicted"] * 1000).round(1)
df8 = df8.rename(columns={"target_freq":"freq_MHz", "time_predicted":"lat_ms", "energy_predicted":"E_J"})
df8 = df8[["kernel", "bucket", "freq_MHz", "lat_ms", "power_W", "E_J"]].sort_values(["bucket","kernel","freq_MHz"])
print("A100-PCIe core-clock sweep (DRAM clock fixed)\n")
df8

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
colors = {"compute": "tab:blue", "memory": "tab:red"}
for kn, g in df8.groupby("kernel"):
    g = g.sort_values("freq_MHz"); b = g["bucket"].iloc[0]
    style = "-o" if b == "compute" else "--s"
    lab = "%s (%s)" % (kn, b)
    ax[0].plot(g["freq_MHz"], g["lat_ms"]/g["lat_ms"].iloc[0], style, color=colors[b], label=lab)
    ax[1].plot(g["freq_MHz"], g["power_W"],                       style, color=colors[b], label=lab)
    ax[2].plot(g["freq_MHz"], g["E_J"]/g["E_J"].iloc[0],          style, color=colors[b], label=lab)
ax[0].set_title("Latency (norm. to 510 MHz)\nsolid=compute drops, dashed=memory flat"); ax[0].set_ylabel("relative latency")
ax[1].set_title("Power (W) - rises for BOTH"); ax[1].set_ylabel("power (W)")
ax[2].set_title("Energy (norm. to 510 MHz)\ncompute ~flat, memory rises = wasted clock"); ax[2].set_ylabel("relative energy")
for a in ax:
    a.set_xlabel("core clock (MHz)"); a.set_xticks([510,705,900]); a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Recap

- **Section 0** used the `gee` Python API directly for one kernel.
- **Sections 1–3** drove `experiments_endtoend/run_estimators.py` over workload JSONs for whole-model energy, DVFS sweeps, and cross-architecture extrapolation.

All of it ran on CPU — no GPU and no profiling. To predict your own model, generate workload JSONs per `test/code/README.md`, drop them in a folder, and point `--workload_folder` at it.